# CardioIA — Treinamento do modelo


**Objetivo.** Treinar um classificador capaz de prever a
feature `pico_risco` a partir da base de dados do material da aula
(`data/base_dados.py`).

## Dependências

O notebook precisa apenas de `numpy`, `pandas`, `scikit-learn` e `joblib`.

In [1]:
# 2.1 - Verificacao das dependencias
import importlib
import subprocess
import sys

REQUISITOS = {
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "joblib": "joblib",
}

faltando = []
for modulo in REQUISITOS:
    try:
        importlib.import_module(modulo)
    except ImportError:
        faltando.append(modulo)

if faltando:
    pacotes = [REQUISITOS[modulo] for modulo in faltando]
    print("Instalando dependencias ausentes:", pacotes)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pacotes], check=False)
else:
    print("Todas as dependencias ja estao disponiveis - nenhuma instalacao necessaria.")

import joblib
import numpy
import pandas
import sklearn

EM_COLAB = "google.colab" in sys.modules
print("python  :", sys.version.split()[0])
print("numpy   :", numpy.__version__)
print("pandas  :", pandas.__version__)
print("sklearn :", sklearn.__version__)
print("joblib  :", joblib.__version__)
print("ambiente:", "Google Colab" if EM_COLAB else "local")

Todas as dependencias ja estao disponiveis - nenhuma instalacao necessaria.
python  : 3.11.7
numpy   : 1.26.4
pandas  : 2.1.4
sklearn : 1.2.2
joblib  : 1.2.0
ambiente: local


## Bootstrap do pacote, imports e semente aleatória

A célula 3.1 localiza a **raiz do projeto** (a pasta que contém `data/base_dados.py` e
`src/cardioia/ml/dataset.py`), insere `src/` no `sys.path` e importa os módulos. 

parâmetros:

- `SEMENTE = 42`;
- `TAMANHO_TESTE = 0.20` — divisão 80/20;
- `COLUNAS_PREDITORAS` — 10 features;
- `LIMIARES_DO_ROTULO` e `CORTE_RISK_SCORE` — usados para checagem

In [ ]:

from pathlib import Path

NOME_DO_REPOSITORIO = "cardioia_Fase6"
URL_DO_REPOSITORIO = "https://github.com/ReMendess/cardioia_Fase6.git"


def parece_raiz_do_projeto(caminho: Path) -> bool:
    # A raiz precisa ter o material oficial E a camada modularizada.
    return (caminho / "data" / "base_dados.py").is_file() and (
        caminho / "src" / "cardioia" / "ml" / "dataset.py"
    ).is_file()


def localizar_raiz(inicio=None):
    base = Path(inicio or Path.cwd()).resolve()
    for candidato in (base, *base.parents):
        if parece_raiz_do_projeto(candidato):
            return candidato
    return None


RAIZ = localizar_raiz()

if RAIZ is None:
    # Plano B: clonar o repositorio (funciona apenas se data/ e src/ estiverem versionados).
    destino = Path("/content") if Path("/content").is_dir() else Path.cwd()
    print("[bootstrap] projeto nao encontrado no ambiente; tentando clonar o repositorio...")
    subprocess.run(
        ["git", "clone", "--depth", "1", URL_DO_REPOSITORIO, str(destino / NOME_DO_REPOSITORIO)],
        check=False,
    )
    RAIZ = localizar_raiz(destino / NOME_DO_REPOSITORIO)

if RAIZ is None:
    raise RuntimeError(
        "Projeto CardioIA nao encontrado. Envie a pasta do projeto antes de continuar:\n"
        "  1) compacte 'cardioia_Fase6/' em .zip;\n"
        "  2) no Colab, rode:  from google.colab import files; files.upload()\n"
        "     e descompacte com zipfile.ZipFile('<arquivo>.zip').extractall('/content')\n"
        "Localmente, execute o notebook a partir da raiz do repositorio."
    )

PASTA_SRC = RAIZ / "src"
if str(PASTA_SRC) not in sys.path:
    sys.path.insert(0, str(PASTA_SRC))

from cardioia.ml import dataset, evaluate, predict, train  # noqa: E402 - depende do sys.path

SEMENTE = dataset.SEMENTE
TAMANHO_TESTE = dataset.TAMANHO_TESTE
COLUNAS_PREDITORAS = list(dataset.COLUNAS_PREDITORAS)
LIMIARES_DO_ROTULO = dict(dataset.LIMIARES_DO_ROTULO)
CORTE_RISK_SCORE = dataset.CORTE_RISK_SCORE
PASTA_MODELOS = train.resolver_pasta_modelos(raiz=RAIZ)

print("Diretorio atual     :", Path.cwd())
print("Raiz do projeto     :", RAIZ)
print("Pasta de modelos    :", PASTA_MODELOS)
print("Semente             :", SEMENTE, "| tamanho do teste:", TAMANHO_TESTE)
print("Features            :", len(COLUNAS_PREDITORAS))
print("Modulos importados  :", ", ".join(["dataset", "evaluate", "predict", "train"]))

Diretorio atual     : c:\Users\Pichau\OneDrive\Área de Trabalho\cardioia_Fase6\notebooks
Raiz do projeto     : C:\Users\Pichau\OneDrive\Área de Trabalho\cardioia_Fase6
Pasta de modelos    : C:\Users\Pichau\OneDrive\Área de Trabalho\cardioia_Fase6\models
Semente             : 42 | tamanho do teste: 0.2
Features            : 10
Modulos importados  : dataset, evaluate, predict, train


## Carregamento ou geração dos dados

In [3]:
# Leitura dos tres arquivos

ARQUIVOS_OFICIAIS = (
    dataset.NOME_ARQUIVO_BASE,
    dataset.NOME_ARQUIVO_CONHECIMENTO,
    dataset.NOME_ARQUIVO_NOVO_PACIENTE,
)

fontes = {}
for nome in ARQUIVOS_OFICIAIS:
    try:
        fontes[nome] = dataset.ler_arquivo_oficial(nome, raiz=RAIZ)
        print(f"[OK]      data/{nome} | {len(fontes[nome])} caracteres")
    except dataset.ArquivoOficialNaoEncontradoError as erro:
        print(f"[AUSENTE] {erro}")

print()
print("Base oficial em:", dataset.caminho_da_base_oficial(raiz=RAIZ))

[OK]      data/base_dados.py | 1006 caracteres
[OK]      data/base_conhecimento.py | 628 caracteres
[OK]      data/novo_paciente.py | 281 caracteres

Base oficial em: C:\Users\Pichau\OneDrive\Área de Trabalho\cardioia_Fase6\data\base_dados.py


In [4]:
#  Geracao da base pelo modulo

REPLICA_BASE_DADOS = r'''
import numpy as np
import pandas as pd

np.random.seed(42)
n = 5000

df = pd.DataFrame({
    "idade_media": np.random.normal(55, 12, n).clip(18, 90),
    "carga_sistema": np.random.uniform(30, 100, n),
    "tempo_espera": np.random.normal(45, 20, n).clip(5, 180),
    "fc_media": np.random.normal(85, 15, n).clip(50, 150),
    "spo2_media": np.random.normal(95, 3, n).clip(70, 100),
    "pa_sistolica": np.random.normal(125, 20, n).clip(80, 200),
    "temperatura": np.random.normal(24, 5, n).clip(10, 40),
    "densidade_eventos": np.random.poisson(3, n),
    "eventos_24h": np.random.poisson(2, n),
    "disponibilidade_recursos": np.random.uniform(20, 100, n)
})

risk_score = (
    (df["carga_sistema"] > 80).astype(int) +
    (df["tempo_espera"] > 60).astype(int) +
    (df["spo2_media"] < 92).astype(int) +
    (df["fc_media"] > 110).astype(int) +
    (df["eventos_24h"] > 4).astype(int) +
    (df["disponibilidade_recursos"] < 40).astype(int)
)

df["pico_risco"] = (risk_score >= 2).astype(int)
'''

try:
    # Caminho normal: executa o arquivo oficial do material da aula.
    df, origem = dataset.carregar_base_oficial(raiz=RAIZ, retornar_origem=True)
except dataset.ArquivoOficialNaoEncontradoError as erro:
    print("[FAILOVER]", erro)
    print("[ATENCAO] A base NAO veio do arquivo oficial - registre este failover no relatorio.")
    df, origem = dataset.carregar_base_oficial(
        codigo_fonte=REPLICA_BASE_DADOS, retornar_origem=True
    )

origem_dados = origem["origem"]
print()
print("origem da base         :", origem_dados)
print("usado failover         :", origem["usado_failover"])
print("dimensoes do DataFrame :", df.shape)
print("sha256 da base         :", dataset.sha256_da_base(df))


origem da base         : arquivo oficial data/base_dados.py
usado failover         : False
dimensoes do DataFrame : (5000, 11)
sha256 da base         : 89f3a65f6e49b1100062f090b7819eacac4bc501a840a3172baf9c8291454275


In [5]:
# Base de conhecimento (protocolos) e simulacao do paciente
if dataset.NOME_ARQUIVO_CONHECIMENTO in fontes:
    espaco = {}
    exec(compile(fontes[dataset.NOME_ARQUIVO_CONHECIMENTO], "data/base_conhecimento.py", "exec"), espaco)
    PROTOCOLOS_AULA = list(espaco["protocolos"])
    print("Protocolos carregados de data/base_conhecimento.py:", len(PROTOCOLOS_AULA))
else:
    PROTOCOLOS_AULA = []
    print("[AVISO] data/base_conhecimento.py ausente - a secao 13 nao tera protocolos.")

for indice, protocolo in enumerate(PROTOCOLOS_AULA, start=1):
    print(f"{indice:2d}. {protocolo}")

print()
if dataset.NOME_ARQUIVO_NOVO_PACIENTE in fontes:
    fonte_paciente = fontes[dataset.NOME_ARQUIVO_NOVO_PACIENTE]
    print("Inspecao estatica de data/novo_paciente.py (arquivo preservado, nao editado):")
    print("  - contem 'import pandas':", "import pandas" in fonte_paciente)
    print("  - usa a coluna 'disponibilidade recursos' (com espaco):", "disponibilidade recursos" in fonte_paciente)
    print("  - usa a coluna 'disponibilidade_recursos' (com underscore):", "disponibilidade_recursos" in fonte_paciente)
    print("  => o fragmento da aula nao executa como esta (falta o import de pandas) e a")
    print("     ultima coluna esta escrita com espaco.")
else:
    print("[AVISO] data/novo_paciente.py ausente - a secao 13 usara o caso descrito no plano.")

# O modulo le o caso oficial com um 'pd' substituto e normaliza o nome da coluna.
caso_paciente = dataset.carregar_caso_do_paciente_da_aula(raiz=RAIZ)
print()
print("Caso do paciente normalizado (disponibilidade recursos -> disponibilidade_recursos):")
for nome, valor in caso_paciente.items():
    print(f"  {nome:28s}: {valor}")

Protocolos carregados de data/base_conhecimento.py: 10
 1. Se SPO2 < 88% → iniciar suporte respiratório imediato
 2. Se carga do sistema > 85% → abrir unidades temporárias
 3. Se FC > 120 → iniciar monitoramento intensivo
 4. Se eventos_24h > 5 → ativar equipe de resposta rápida
 5. Se disponibilidade_recursos < 30% → redistribuir insumos
 6. Se tempo_espera > 90 min → priorizar triagem emergencial
 7. Se temperatura > 38°C média → iniciar protocolo infeccioso
 8. Se densidade_eventos > 6 → ativar plano de contingência
 9. Se PA sistólica < 90 → iniciar protocolo de choque
10. Se risco sistêmico alto → ativar estado de emergência

Inspecao estatica de data/novo_paciente.py (arquivo preservado, nao editado):
  - contem 'import pandas': False
  - usa a coluna 'disponibilidade recursos' (com espaco): True
  - usa a coluna 'disponibilidade_recursos' (com underscore): False
  => o fragmento da aula nao executa como esta (falta o import de pandas) e a
     ultima coluna esta escrita com espa

## Inspeção da base

Tratamento dos dados, para avaliar se a base realmente contém: valores ausentes e
duplicatas. Tiramos a estatísticas descritivas, distribuição do alvo `pico_risco` e a distribuição do
`risk_score` que deu origem ao rótulo.

In [6]:
# Validacao e inspecao da base
resumo = dataset.validar_base(df)
print("Dimensoes:", resumo["dimensoes"])
print()
print("Tipos, distintos e ausentes por coluna:")
for coluna in df.columns:
    ausentes = resumo["valores_ausentes_por_coluna"].get(coluna, 0)
    print(f"  {coluna:28s} {resumo['tipos'][coluna]:8s} distintos={df[coluna].nunique():5d} ausentes={ausentes}")
print()
print("Total de valores ausentes :", resumo["valores_ausentes_total"])
print("Linhas duplicadas         :", resumo["linhas_duplicadas"])
print("Avisos da validacao       :", resumo["avisos"] if resumo["avisos"] else "nenhum")
print()
print("Estatisticas descritivas:")
print(df.describe().round(3).to_string())

Dimensoes: {'linhas': 5000, 'colunas': 11}

Tipos, distintos e ausentes por coluna:
  idade_media                  float64  distintos= 4987 ausentes=0
  carga_sistema                float64  distintos= 5000 ausentes=0
  tempo_espera                 float64  distintos= 4892 ausentes=0
  fc_media                     float64  distintos= 4946 ausentes=0
  spo2_media                   float64  distintos= 4763 ausentes=0
  pa_sistolica                 float64  distintos= 4944 ausentes=0
  temperatura                  float64  distintos= 4979 ausentes=0
  densidade_eventos            int32    distintos=   13 ausentes=0
  eventos_24h                  int32    distintos=   10 ausentes=0
  disponibilidade_recursos     float64  distintos= 5000 ausentes=0
  pico_risco                   int32    distintos=    2 ausentes=0

Total de valores ausentes : 0
Linhas duplicadas         : 0
Avisos da validacao       : nenhum

Estatisticas descritivas:
       idade_media  carga_sistema  tempo_espera  fc_medi

In [8]:
# Distribuicao da variavel-alvo
contagens = df[dataset.VARIAVEL_ALVO].value_counts().sort_index()
proporcoes = df[dataset.VARIAVEL_ALVO].value_counts(normalize=True).sort_index()

distribuicao = pandas.DataFrame({"contagem": contagens, "proporcao": proporcoes.round(4)})
distribuicao.index.name = dataset.VARIAVEL_ALVO
print("Distribuicao de", dataset.VARIAVEL_ALVO + ":")
print(distribuicao.to_string())
print()
print("Classe minoritaria (pico_risco=1):", f"{proporcoes.get(1, 0):.2%}", "da base")
print("Leitura: ha desbalanceamento moderado. A acuracia isolada nao basta -")
print("por isso o recall e a precisao da classe 1 tambem serao reportados (secao 9).")
print()


risk_score_recalculado = dataset.calcular_risk_score(df)
pico_risco_recalculado = dataset.rotulo_pela_regra(df)

print("Distribuicao do risk_score reconstruido:")
print(risk_score_recalculado.value_counts().sort_index().to_string())
print()
concordancia = float((pico_risco_recalculado == df[dataset.VARIAVEL_ALVO]).mean())
print("Concordancia regra reconstruida x coluna pico_risco:", concordancia)
assert concordancia == 1.0, "A regra reconstruida NAO reproduz o rotulo oficial."
print("[OK] Os limiares do modulo correspondem exatamente ao material oficial.")
print()
print("Frequencia dos limiares TEXTUAIS da base de conhecimento da aula,")
print("que sao DIFERENTES dos limiares do rotulo:")
efeito_dos_protocolos = pandas.Series({
    "SPO2 < 88": int((df["spo2_media"] < 88).sum()),
    "carga do sistema > 85": int((df["carga_sistema"] > 85).sum()),
    "FC > 120": int((df["fc_media"] > 120).sum()),
    "eventos_24h > 5": int((df["eventos_24h"] > 5).sum()),
    "disponibilidade_recursos < 30": int((df["disponibilidade_recursos"] < 30).sum()),
    "tempo_espera > 90": int((df["tempo_espera"] > 90).sum()),
    "temperatura > 38": int((df["temperatura"] > 38).sum()),
    "densidade_eventos > 6": int((df["densidade_eventos"] > 6).sum()),
    "PA sistolica < 90": int((df["pa_sistolica"] < 90).sum()),
})
print(efeito_dos_protocolos.to_string())

Distribuicao de pico_risco:
            contagem  proporcao
pico_risco                     
0               3671     0.7342
1               1329     0.2658

Classe minoritaria (pico_risco=1): 26.58% da base
Leitura: ha desbalanceamento moderado. A acuracia isolada nao basta -
por isso o recall e a precisao da classe 1 tambem serao reportados (secao 9).

Distribuicao do risk_score reconstruido:
0    1601
1    2070
2    1029
3     262
4      36
5       2

Concordancia regra reconstruida x coluna pico_risco: 1.0
[OK] Os limiares do modulo correspondem exatamente ao material oficial.

Frequencia dos limiares TEXTUAIS da base de conhecimento da aula,
que sao DIFERENTES dos limiares do rotulo:
SPO2 < 88                          48
carga do sistema > 85            1034
FC > 120                           47
eventos_24h > 5                    91
disponibilidade_recursos < 30     584
tempo_espera > 90                  57
temperatura > 38                   14
densidade_eventos > 6             149

## Preparação das variáveis independetes e da variável alvo `pico_risco`

- **`X`**: as 10 colunas do gerador.
- **`y`**: a coluna `pico_risco`.


In [11]:
# Separacao X/y pelo modulo + verificacoes anti-vazamento
X, y = dataset.separar_features_e_alvo(df)

assert set(COLUNAS_PREDITORAS).issubset(df.columns)
assert dataset.VARIAVEL_ALVO not in COLUNAS_PREDITORAS
assert "risk_score" not in X.columns, "Vazamento: coluna derivada do alvo entrou em X."
assert not [c for c in X.columns if "risk" in c]
assert X.isna().sum().sum() == 0 and y.isna().sum() == 0

print("X:", X.shape, "| y:", y.shape)
print("Features (ordem preservada do material oficial):")
for posicao, coluna in enumerate(X.columns, start=1):
    print(f"  {posicao:2d}. {coluna:28s} {str(X[coluna].dtype)}")
print()
print("Variavel-alvo:", dataset.VARIAVEL_ALVO, "| valores possiveis:", sorted(y.unique().tolist()))
print("Colunas derivadas do alvo excluidas:", list(dataset.COLUNAS_AUXILIARES_IGNORADAS))
print("Valores ausentes em X:", int(X.isna().sum().sum()), "| em y:", int(y.isna().sum()))

X: (5000, 10) | y: (5000,)
Features (ordem preservada do material oficial):
   1. idade_media                  float64
   2. carga_sistema                float64
   3. tempo_espera                 float64
   4. fc_media                     float64
   5. spo2_media                   float64
   6. pa_sistolica                 float64
   7. temperatura                  float64
   8. densidade_eventos            int32
   9. eventos_24h                  int32
  10. disponibilidade_recursos     float64

Variavel-alvo: pico_risco | valores possiveis: [0, 1]
Colunas derivadas do alvo excluidas: ['risk_score']
Valores ausentes em X: 0 | em y: 0


## Separação de treino e teste 
`dataset.divisao_treino_teste` é a **única** fonte da partição: 80% treino / 20% teste,


In [14]:

X_treino, X_teste, y_treino, y_teste = dataset.divisao_treino_teste(X, y)

resumo_split = pandas.DataFrame({
    "amostras": [len(y_treino), len(y_teste)],
    "positivos": [int(y_treino.sum()), int(y_teste.sum())],
    "proporcao_positivos": [float(y_treino.mean()), float(y_teste.mean())],
}, index=["treino", "teste"])
print(resumo_split.round(4).to_string())
print()
assert set(X_treino.index).isdisjoint(set(X_teste.index)), "Vazamento entre treino e teste."
assert len(X_treino) + len(X_teste) == len(X)
print("[OK] Conjuntos disjuntos e base totalmente alocada.")
print("Diferenca de proporcao treino/teste:",
      round(abs(float(y_treino.mean()) - float(y_teste.mean())), 6),
      "(estratificacao preservou o balanco)")

        amostras  positivos  proporcao_positivos
treino      4000       1063               0.2658
teste       1000        266               0.2660

[OK] Conjuntos disjuntos e base totalmente alocada.
Diferenca de proporcao treino/teste: 0.00025 (estratificacao preservou o balanco)


## Treinamento 

O treino é executado por `train.treinar_e_avaliar`

| Modelo | Hiperparâmetros |
|---|---|
| **Regressão Logística** | `make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))` |
| **Random Forest** | `RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)` |

Regressão Logística por que é interpretável e serve de referência para dados tabulares;

Random Forest é não linear por construção e representa bem regras com limiares, como as do rótulo

**Mesmas condições:** mesmo `X_treino`/`y_treino`, mesma avaliação no mesmo `X_teste`.

> **Critério de escolha do modelo**
> 1. maior **recall da classe `pico_risco=1`** (um falso negativo é o erro mais custoso);
> 2. desempate por **acurácia**;
> 3. persistindo o empate, o modelo mais **interpretável**.

In [15]:
# mesmo treino para os dois modelos
execucao = train.treinar_e_avaliar(
    df=df,
    semente=SEMENTE,
    tamanho_teste=TAMANHO_TESTE,
    n_estimators=train.N_ESTIMATORS_PADRAO,
    raiz=RAIZ,
    origem_da_base=origem_dados,
)

print("Modelos treinados:")
for nome, modelo in execucao["modelos"].items():
    print(f"  [treinado] {nome:22s} | {type(modelo).__name__}")
print()
print("Hiperparametros efetivamente usados (lidos do objeto treinado):")
import json as _json
for nome, modelo in execucao["modelos"].items():
    print(f"  {nome}: {_json.dumps(train.descrever_parametros(modelo), ensure_ascii=False)}")
print()
print("Criterio de escolha:", execucao["metadados"]["criterio_de_escolha"])

Modelos treinados:
  [treinado] Regressao Logistica    | Pipeline
  [treinado] Random Forest          | RandomForestClassifier

Hiperparametros efetivamente usados (lidos do objeto treinado):
  Regressao Logistica: {"tipo": "Pipeline", "passos": [{"passo": "standardscaler", "classe": "StandardScaler", "parametros": {}}, {"passo": "logisticregression", "classe": "LogisticRegression", "parametros": {"max_iter": 1000, "random_state": 42, "n_jobs": null, "C": 1.0, "penalty": "l2", "solver": "lbfgs"}}]}
  Random Forest: {"tipo": "RandomForestClassifier", "parametros": {"random_state": 42, "n_estimators": 300, "n_jobs": -1, "max_depth": null}}

Criterio de escolha: maior recall da classe pico_risco=1 no conjunto de teste; desempate por acuracia; depois por interpretabilidade


## Avaliação: acurácia, matriz de confusão e relatório de classificação

As métricas vêm de `evaluate.avaliar_modelo` (que usa apenas funções do `scikit-learn`) sobre o
conjunto de teste.

In [16]:
# 9.1 - Metricas no conjunto de teste (modulo evaluate)
for nome, avaliacao in execucao["avaliacoes"].items():
    print(evaluate.formatar_relatorio(avaliacao))
    print()

print("=" * 72)
print("Comparativo direto no MESMO conjunto de teste:")
comparativo = pandas.DataFrame([
    {
        "acuracia": avaliacao["acuracia"],
        "recall_classe_1": avaliacao["recall_classe_1"],
        "precisao_classe_1": avaliacao["precisao_classe_1"],
        "f1_classe_1": avaliacao["f1_classe_1"],
    }
    for avaliacao in execucao["avaliacoes"].values()
], index=list(execucao["avaliacoes"]))
print(comparativo.round(4).to_string())

Regressao Logistica
Acuracia: 0.816

Matriz de confusao (linhas = real, colunas = previsto, ordem [0, 1]):
                   previsto 0  previsto 1
real pico_risco=0         669          65
real pico_risco=1         119         147

Relatorio de classificacao:
              precision    recall  f1-score   support

pico_risco=0       0.85      0.91      0.88       734
pico_risco=1       0.69      0.55      0.62       266

    accuracy                           0.82      1000
   macro avg       0.77      0.73      0.75      1000
weighted avg       0.81      0.82      0.81      1000
Leitura: TN = 669 | FP = 65 | FN = 119 | TP = 147

Random Forest
Acuracia: 0.999

Matriz de confusao (linhas = real, colunas = previsto, ordem [0, 1]):
                   previsto 0  previsto 1
real pico_risco=0         734           0
real pico_risco=1           1         265

Relatorio de classificacao:
              precision    recall  f1-score   support

pico_risco=0       1.00      1.00      1.00       

In [19]:
regra_no_teste = execucao["regra_deterministica_no_teste"]
avaliacao_da_regra = evaluate.avaliar_predicoes(
    y_teste, regra_no_teste, nome_do_modelo="Regra deterministica"
)
print(evaluate.formatar_relatorio(avaliacao_da_regra))
print()
print("Acuracia da regra (teto de referencia):", execucao["acuracia_da_regra"])
print("Observacao: acuracia = 1 e consequencia de o rotulo ter sido DEFINIDO por essa regra.")
print("O numero relevante e a distancia entre cada modelo e esse teto.")

Regra deterministica
Acuracia: 1.0

Matriz de confusao (linhas = real, colunas = previsto, ordem [0, 1]):
                   previsto 0  previsto 1
real pico_risco=0         734           0
real pico_risco=1           0         266

Relatorio de classificacao:
              precision    recall  f1-score   support

pico_risco=0       1.00      1.00      1.00       734
pico_risco=1       1.00      1.00      1.00       266

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000
Leitura: TN = 734 | FP = 0 | FN = 0 | TP = 266

Acuracia da regra (teto de referencia): 1.0
Observacao: acuracia = 1 e consequencia de o rotulo ter sido DEFINIDO por essa regra.
O numero relevante e a distancia entre cada modelo e esse teto.


## Análise dos resultados 

In [ ]:
#  Criterio pre-declarado aplicado + o que o modelo aprendeu
comparativo_ordenado = pandas.DataFrame(execucao["comparativo"]).set_index("modelo")
print("Ordenacao pelo criterio pre-declarado (1o recall da classe 1, 2o acuracia):")
print(comparativo_ordenado[[
    "acuracia", "recall_classe_1", "precisao_classe_1", "f1_classe_1", "interpretabilidade"
]].round(4).to_string())
modelo_escolhido_nome = execucao["nome_modelo_escolhido"]
modelo_escolhido = execucao["modelo_escolhido"]
print()
print("Modelo escolhido:", modelo_escolhido_nome)
print()

floresta = execucao["modelos"][train.NOME_MODELO_NAO_LINEAR]
importancias = pandas.Series(
    floresta.feature_importances_, index=COLUNAS_PREDITORAS
).sort_values(ascending=False)
print("Importancia das variaveis (Random Forest, impureza de Gini):")
print(importancias.round(4).to_string())
print()

regressao = execucao["modelos"][train.NOME_MODELO_INTERPRETAVEL].named_steps["logisticregression"]
coeficientes = pandas.Series(regressao.coef_[0], index=COLUNAS_PREDITORAS)
coeficientes = coeficientes.reindex(coeficientes.abs().sort_values(ascending=False).index)
print("Coeficientes da Regressao Logistica (features padronizadas; sinal = direcao do risco):")
print(coeficientes.round(4).to_string())
print()

nulas = importancias[importancias < 0.01]
if len(nulas):
    print("Variaveis praticamente ignoradas pelo Random Forest (< 1% de importancia):")
    print(nulas.round(6).to_string())
else:
    print("Nenhuma variavel ficou abaixo de 1% de importancia no Random Forest.")
print()

erros = X_teste.copy()
erros["pico_risco_real"] = y_teste.values
erros["pico_risco_previsto"] = modelo_escolhido.predict(X_teste)
erros["risk_score_da_regra"] = risk_score_recalculado.loc[X_teste.index].values
falsos_negativos = erros[(erros["pico_risco_real"] == 1) & (erros["pico_risco_previsto"] == 0)]
falsos_positivos = erros[(erros["pico_risco_real"] == 0) & (erros["pico_risco_previsto"] == 1)]

print(f"Erros do modelo escolhido ({modelo_escolhido_nome}):")
print("  falsos negativos (risco nao sinalizado):", len(falsos_negativos))
print("  falsos positivos (alarme sem risco real):", len(falsos_positivos))
if len(falsos_negativos):
    print()
    print("Exemplos de falsos negativos (com o risk_score da regra oficial):")
    print(falsos_negativos.head(5).round(2).to_string())

Ordenacao pelo criterio pre-declarado (1o recall da classe 1, 2o acuracia):
                     acuracia  recall_classe_1  precisao_classe_1  f1_classe_1  interpretabilidade
modelo                                                                                            
Random Forest           0.999           0.9962             1.0000       0.9981                   1
Regressao Logistica     0.816           0.5526             0.6934       0.6151                   0

Modelo escolhido: Random Forest

Importancia das variaveis (Random Forest, impureza de Gini):
carga_sistema               0.2287
disponibilidade_recursos    0.2272
tempo_espera                0.2043
spo2_media                  0.1654
fc_media                    0.0695
eventos_24h                 0.0635
idade_media                 0.0132
temperatura                 0.0118
pa_sistolica                0.0114
densidade_eventos           0.0051

Coeficientes da Regressao Logistica (features padronizadas; sinal = direcao do ris

In [21]:
# Divergencias entre o rotulo e os protocolos  + resumo do experimento
divergencias = pandas.DataFrame([
    ("spo2_media", "rotulo: < 92", "protocolo: < 88"),
    ("carga_sistema", "rotulo: > 80", "protocolo: > 85"),
    ("fc_media", "rotulo: > 110", "protocolo: > 120"),
    ("eventos_24h", "rotulo: > 4", "protocolo: > 5"),
    ("disponibilidade_recursos", "rotulo: < 40", "protocolo: < 30"),
    ("tempo_espera", "rotulo: > 60", "protocolo: > 90"),
    ("temperatura", "NAO entra no rotulo", "protocolo: > 38 (no gerador a variavel e N(24,5))"),
    ("densidade_eventos", "NAO entra no rotulo", "protocolo: > 6"),
    ("pa_sistolica", "NAO entra no rotulo", "protocolo: < 90"),
    ("risco sistemico", "rotulo: risk_score >= 2", "protocolo: 'risco sistemico alto' (sem limiar numerico)"),
], columns=["variavel", "limiar_no_rotulo_pico_risco", "limiar_no_protocolo_da_aula"])

print("Divergencias entre os limiares do rotulo e os protocolos textuais da aula:")
print(divergencias.to_string(index=False))
print()
print("Implicacao: a camada de ML (rotulo) e a camada de protocolos sao SEPARADAS.")
print("O modelo aprende o rotulo; os protocolos continuam sendo avaliados de forma")
print("deterministica em Python (secao 13). Um LLM nao decide esses limiares.")
print()
print(train.formatar_resumo(execucao))

Divergencias entre os limiares do rotulo e os protocolos textuais da aula:
                variavel limiar_no_rotulo_pico_risco                             limiar_no_protocolo_da_aula
              spo2_media                rotulo: < 92                                         protocolo: < 88
           carga_sistema                rotulo: > 80                                         protocolo: > 85
                fc_media               rotulo: > 110                                        protocolo: > 120
             eventos_24h                 rotulo: > 4                                          protocolo: > 5
disponibilidade_recursos                rotulo: < 40                                         protocolo: < 30
            tempo_espera                rotulo: > 60                                         protocolo: > 90
             temperatura         NAO entra no rotulo       protocolo: > 38 (no gerador a variavel e N(24,5))
       densidade_eventos         NAO entra no rotulo 

## Salvando o modelo treinado

`train.salvar_artefatos` grava dois arquivos na pasta `models/`:

| Arquivo | Conteúdo |
|---|---|
| `modelo_pico_risco.pkl` | O modelo escolhido, serializado com `joblib`. |
| `modelo_pico_risco_metadata.json` | Metadados de rastreabilidade: features (e a **ordem**), semente, split, critério de escolha, parâmetros do estimador, limiar de decisão, métricas de teste, matriz de confusão, faixas observadas, versões das bibliotecas, SHA-256 da base usada e o aviso de dados sintéticos. |

O JSON é o que permite reutilizar o modelo com segurança (inclusive pelos agentes, na próxima
etapa): quem carregar o `.pkl` precisa saber exatamente quais colunas, em que ordem, o modelo
espera — e de onde vieram os números, sem inventar probabilidades.

In [22]:
# Salvamento do modelo e dos metadados (modulo train)
artefatos = train.salvar_artefatos(execucao, pasta_modelos=PASTA_MODELOS, raiz=RAIZ)
caminho_modelo = artefatos["caminho_modelo"]
caminho_metadados = artefatos["caminho_metadados"]

print("Artefatos gravados:")
print("  modelo   :", caminho_modelo, "|", artefatos["tamanho_modelo_bytes"], "bytes")
print("  metadados:", caminho_metadados, "|", artefatos["tamanho_metadados_bytes"], "bytes")
print()
import json as _json
print("Metadados (sem o bloco de metricas, ja impresso na secao 9):")
print(_json.dumps(
    {chave: valor for chave, valor in artefatos["metadados"].items() if chave != "metricas_no_teste"},
    ensure_ascii=False, indent=2,
))

Artefatos gravados:
  modelo   : C:\Users\Pichau\OneDrive\Área de Trabalho\cardioia_Fase6\models\modelo_pico_risco.pkl | 4251049 bytes
  metadados: C:\Users\Pichau\OneDrive\Área de Trabalho\cardioia_Fase6\models\modelo_pico_risco_metadata.json | 3251 bytes

Metadados (sem o bloco de metricas, ja impresso na secao 9):
{
  "nome_do_modelo": "Random Forest",
  "classe_do_estimador": "RandomForestClassifier",
  "variavel_alvo": "pico_risco",
  "features": [
    "idade_media",
    "carga_sistema",
    "tempo_espera",
    "fc_media",
    "spo2_media",
    "pa_sistolica",
    "temperatura",
    "densidade_eventos",
    "eventos_24h",
    "disponibilidade_recursos"
  ],
  "colunas_derivadas_do_alvo_excluidas": [
    "risk_score"
  ],
  "criterio_de_escolha": "maior recall da classe pico_risco=1 no conjunto de teste; desempate por acuracia; depois por interpretabilidade",
  "parametros_do_estimador": {
    "tipo": "RandomForestClassifier",
    "parametros": {
      "random_state": 42,
      "n_

## Carregamento do modelo salvo e teste de uma nova previsão

O artefato é reutilizado pela função `predict.prever_pico_risco`, que é a **mesma** interface que o
Agente Analista de Risco usará como *tool* na próxima etapa. Ela:

- carrega `models/modelo_pico_risco.pkl` e o JSON de metadados (erro claro se não existirem);
- valida o contrato de entrada (campos ausentes/desconhecidos, texto, booleano, `NaN`/`inf`,
  negativos) e **reordena** as colunas conforme o modelo espera;
- devolve `pico_risco_previsto`, `probabilidade_risco` (valor **cru**, sem arredondar),
  `classificacao_risco`, `modelo_utilizado` e `limiar_decisao`.

O limiar (0,5) é um parâmetro **operacional** do protótipo

In [23]:

modelo_recarregado = predict.carregar_modelo(pasta_modelos=PASTA_MODELOS)
metadados = predict.carregar_metadados(pasta_modelos=PASTA_MODELOS)
print("Modelo recarregado:", type(modelo_recarregado).__name__, "| nome no metadata:", metadados["nome_do_modelo"])

predicoes_em_memoria = modelo_escolhido.predict(X_teste)
predicoes_do_disco = modelo_recarregado.predict(X_teste)
identicas = bool((predicoes_em_memoria == predicoes_do_disco).all())
print("Previsoes identicas antes/depois do salvamento:", identicas)
assert identicas, "O modelo recarregado diverge do modelo em memoria."


def regra_do_rotulo(caso):
    # Implementacao deterministica (mesma de data/base_dados.py) via modulo dataset.
    ativos = dataset.indicadores_ativos(caso)
    risk_score = len(ativos)
    return risk_score, int(risk_score >= CORTE_RISK_SCORE), ativos


def apresentar_caso(caso, titulo):
    resultado = predict.prever_pico_risco(caso, pasta_modelos=PASTA_MODELOS)
    risk_score, pico_da_regra, ativos = regra_do_rotulo(caso)
    print(titulo)
    print("  entrada                :", caso)
    print("  probabilidade (crua)   :", resultado["probabilidade_risco"])
    print("  probabilidade (exibida):", round(resultado["probabilidade_risco"], 4))
    print("  pico_risco (modelo)    :", resultado["pico_risco_previsto"],
          "->", resultado["classificacao_risco"])
    print("  risk_score (regra)     :", risk_score, "-> pico_risco (regra):", pico_da_regra)
    print("  indicadores ativos     :", ativos if ativos else "nenhum")
    print("  modelo e regra concordam:", resultado["pico_risco_previsto"] == pico_da_regra)
    print("  avisos                 :", resultado["avisos"] if resultado["avisos"] else "nenhum")
    print()
    return resultado


caso_baixo = {
    "idade_media": 40, "carga_sistema": 45, "tempo_espera": 20, "fc_media": 78,
    "spo2_media": 98, "pa_sistolica": 118, "temperatura": 22,
    "densidade_eventos": 2, "eventos_24h": 1, "disponibilidade_recursos": 80,
}
caso_alto = {
    "idade_media": 70, "carga_sistema": 88, "tempo_espera": 65, "fc_media": 115,
    "spo2_media": 90, "pa_sistolica": 105, "temperatura": 30,
    "densidade_eventos": 5, "eventos_24h": 6, "disponibilidade_recursos": 35,
}

print("Teste de dois casos novos com o modelo RECARREGADO do disco")
print("=" * 72)
apresentar_caso(caso_baixo, "Caso A - cenario sem sobrecarga (esperado: sem pico de risco)")
apresentar_caso(caso_alto, "Cenario B - cenario critico (esperado: pico de risco)")

Modelo recarregado: RandomForestClassifier | nome no metadata: Random Forest
Previsoes identicas antes/depois do salvamento: True
Teste de dois casos novos com o modelo RECARREGADO do disco
Caso A - cenario sem sobrecarga (esperado: sem pico de risco)
  entrada                : {'idade_media': 40, 'carga_sistema': 45, 'tempo_espera': 20, 'fc_media': 78, 'spo2_media': 98, 'pa_sistolica': 118, 'temperatura': 22, 'densidade_eventos': 2, 'eventos_24h': 1, 'disponibilidade_recursos': 80}
  probabilidade (crua)   : 0.0
  probabilidade (exibida): 0.0
  pico_risco (modelo)    : 0 -> sem_pico_de_risco
  risk_score (regra)     : 0 -> pico_risco (regra): 0
  indicadores ativos     : nenhum
  modelo e regra concordam: True
  avisos                 : nenhum

Cenario B - cenario critico (esperado: pico de risco)
  entrada                : {'idade_media': 70, 'carga_sistema': 88, 'tempo_espera': 65, 'fc_media': 115, 'spo2_media': 90, 'pa_sistolica': 105, 'temperatura': 30, 'densidade_eventos': 5, 'ev

{'pico_risco_previsto': 1,
 'probabilidade_risco': 0.9933333333333333,
 'classificacao_risco': 'pico_de_risco',
 'modelo_utilizado': 'Random Forest',
 'classe_do_estimador': 'RandomForestClassifier',
 'variavel_alvo': 'pico_risco',
 'limiar_decisao': 0.5,
 'regra_de_decisao': 'probabilidade_risco >= limiar (0.5)',
 'features_utilizadas': ['idade_media',
  'carga_sistema',
  'tempo_espera',
  'fc_media',
  'spo2_media',
  'pa_sistolica',
  'temperatura',
  'densidade_eventos',
  'eventos_24h',
  'disponibilidade_recursos'],
 'entrada': {'idade_media': 70.0,
  'carga_sistema': 88.0,
  'tempo_espera': 65.0,
  'fc_media': 115.0,
  'spo2_media': 90.0,
  'pa_sistolica': 105.0,
  'temperatura': 30.0,
  'densidade_eventos': 5.0,
  'eventos_24h': 6.0,
  'disponibilidade_recursos': 35.0},
 'classe_prevista_pelo_estimador': 1,
 'origem_do_modelo': {'caminho_modelo': 'C:\\Users\\Pichau\\OneDrive\\Área de Trabalho\\cardioia_Fase6\\models\\modelo_pico_risco.pkl',
  'semente': 42,
  'sha256_da_base_u

## Simulação com os dados do paciente 

O caso vem de `data/novo_paciente.py`, lido por `dataset.carregar_caso_do_paciente_da_aula`:

| Campo (como está no arquivo) | Valor |
|---|---:|
| `idade_media` | 62 |
| `carga_sistema` | 92 |
| `tempo_espera` | 75 |
| `fc_media` | 118 |
| `spo2_media` | 89 |
| `pa_sistolica` | 100 |
| `temperatura` | 37.5 |
| `densidade_eventos` | 7 |
| `eventos_24h` | 5 |
| `disponibilidade recursos` | 25 |


In [24]:

CONDICOES_PROTOCOLOS = [
    ("spo2 < 88", lambda caso: caso["spo2_media"] < 88),
    ("carga do sistema > 85", lambda caso: caso["carga_sistema"] > 85),
    ("fc > 120", lambda caso: caso["fc_media"] > 120),
    ("eventos_24h > 5", lambda caso: caso["eventos_24h"] > 5),
    ("disponibilidade_recursos < 30", lambda caso: caso["disponibilidade_recursos"] < 30),
    ("tempo_espera > 90", lambda caso: caso["tempo_espera"] > 90),
    ("temperatura > 38", lambda caso: caso["temperatura"] > 38),
    ("densidade_eventos > 6", lambda caso: caso["densidade_eventos"] > 6),
    ("pa sistólica < 90", lambda caso: caso["pa_sistolica"] < 90),
    ("risco sistêmico alto", None),
]

print("Vinculacao entre os protocolos oficiais e as condicoes implementadas:")
vinculacao = []
for protocolo in PROTOCOLOS_AULA:
    casados = [rotulo for rotulo, _ in CONDICOES_PROTOCOLOS if rotulo in protocolo.lower()]
    vinculacao.append((protocolo, len(casados)))
    print(f"  {len(casados)} condicao(oes) -> {protocolo}")
assert all(total == 1 for _, total in vinculacao), "Algum protocolo oficial nao tem (ou tem mais de uma) condicao implementada."
print("[OK] Todos os protocolos oficiais estao implementados deterministicamente.")


def protocolos_acionados(caso, pico_risco_previsto):
    # Aplica, em Python puro, todas as regras da base de conhecimento da aula.
    ativos = []
    for protocolo in PROTOCOLOS_AULA:
        for rotulo, condicao in CONDICOES_PROTOCOLOS:
            if rotulo not in protocolo.lower():
                continue
            if condicao is None:
                dispara = (pico_risco_previsto == 1)
            else:
                dispara = bool(condicao(caso))
            if dispara:
                ativos.append(protocolo)
            break
    return ativos

Vinculacao entre os protocolos oficiais e as condicoes implementadas:
  1 condicao(oes) -> Se SPO2 < 88% → iniciar suporte respiratório imediato
  1 condicao(oes) -> Se carga do sistema > 85% → abrir unidades temporárias
  1 condicao(oes) -> Se FC > 120 → iniciar monitoramento intensivo
  1 condicao(oes) -> Se eventos_24h > 5 → ativar equipe de resposta rápida
  1 condicao(oes) -> Se disponibilidade_recursos < 30% → redistribuir insumos
  1 condicao(oes) -> Se tempo_espera > 90 min → priorizar triagem emergencial
  1 condicao(oes) -> Se temperatura > 38°C média → iniciar protocolo infeccioso
  1 condicao(oes) -> Se densidade_eventos > 6 → ativar plano de contingência
  1 condicao(oes) -> Se PA sistólica < 90 → iniciar protocolo de choque
  1 condicao(oes) -> Se risco sistêmico alto → ativar estado de emergência
[OK] Todos os protocolos oficiais estao implementados deterministicamente.


In [25]:
# 13.2 - Previsao do paciente da aula: modelo salvo + oraculo + protocolos
resultado_paciente = predict.prever_pico_risco(caso_paciente, pasta_modelos=PASTA_MODELOS)
risk_score_paciente, pico_risco_da_regra, indicadores = regra_do_rotulo(caso_paciente)

print("=" * 72)
print("SIMULACAO - Paciente do material da aula")
print("=" * 72)
print("Entrada (valores de data/novo_paciente.py, coluna normalizada):")
for nome, valor in caso_paciente.items():
    print(f"  {nome:28s}: {valor}")
print()
print("Saida do modelo salvo (models/modelo_pico_risco.pkl):")
print("  modelo                  :", resultado_paciente["modelo_utilizado"],
      "|", resultado_paciente["classe_do_estimador"])
print("  probabilidade (crua)    :", resultado_paciente["probabilidade_risco"])
print("  probabilidade (exibida) :", round(resultado_paciente["probabilidade_risco"], 4))
print("  pico_risco previsto     :", resultado_paciente["pico_risco_previsto"],
      "->", resultado_paciente["classificacao_risco"])
print("  limiar e regra de decisao:", resultado_paciente["limiar_decisao"], "|",
      resultado_paciente["regra_de_decisao"])
print()
print("Verificacao deterministica (regra de data/base_dados.py):")
print("  indicadores ativos      :", indicadores)
print("  risk_score              :", risk_score_paciente, "de 6")
print("  pico_risco (regra)      :", pico_risco_da_regra)
print("  o modelo e a regra concordam:", resultado_paciente["pico_risco_previsto"] == pico_risco_da_regra)
print()
protocolos = protocolos_acionados(caso_paciente, resultado_paciente["pico_risco_previsto"])
print("Protocolos da base de conhecimento acionados (", len(protocolos), "):")
for protocolo in protocolos:
    print("  -", protocolo)
print()
print("Resumo pronto para um agente (formato de retorno de tool):")
print(" ", predict.resumo_para_agente(resultado_paciente))
print()
print("Observacao: o protocolo de temperatura (> 38) nao dispara porque a temperatura informada")
print("e 37.5 - e no gerador a variavel e N(24,5), ou seja, essa regra quase nunca e ativada")
print("pela base sintetica. Registro mantido por fidelidade ao material.")
print()
print("AVISO:", resultado_paciente["aviso_de_uso"])

SIMULACAO - Paciente do material da aula
Entrada (valores de data/novo_paciente.py, coluna normalizada):
  idade_media                 : 62.0
  carga_sistema               : 92.0
  tempo_espera                : 75.0
  fc_media                    : 118.0
  spo2_media                  : 89.0
  pa_sistolica                : 100.0
  temperatura                 : 37.5
  densidade_eventos           : 7.0
  eventos_24h                 : 5.0
  disponibilidade_recursos    : 25.0

Saida do modelo salvo (models/modelo_pico_risco.pkl):
  modelo                  : Random Forest | RandomForestClassifier
  probabilidade (crua)    : 0.99
  probabilidade (exibida) : 0.99
  pico_risco previsto     : 1 -> pico_de_risco
  limiar e regra de decisao: 0.5 | probabilidade_risco >= limiar (0.5)

Verificacao deterministica (regra de data/base_dados.py):
  indicadores ativos      : ['carga_sistema > 80', 'tempo_espera > 60', 'spo2_media < 92', 'fc_media > 110', 'eventos_24h > 4', 'disponibilidade_recursos < 40']

## Salvando o zip do Google Colab


In [ ]:
# 14.1 - Empacotamento/download dos artefatos + resumo final
from zipfile import ZipFile

arquivos_gerados = sorted(caminho for caminho in PASTA_MODELOS.glob("*") if caminho.is_file())
print("Artefatos em", PASTA_MODELOS, ":")
for caminho in arquivos_gerados:
    print(f"  {caminho.name:38s} {caminho.stat().st_size:>8d} bytes")
print()

if EM_COLAB:
    caminho_zip = (PASTA_MODELOS.parent / "cardioia_fase6_modelo.zip").resolve()
    with ZipFile(caminho_zip, "w") as pacote:
        for caminho in arquivos_gerados:
            pacote.write(caminho, arcname=f"models/{caminho.name}")
    print("Pacote criado:", caminho_zip, "|", caminho_zip.stat().st_size, "bytes")
    print("Iniciando o download...")
    from google.colab import files  # apenas disponivel no Colab
    files.download(str(caminho_zip))
else:
    print("Execucao local: os artefatos permanecem em", PASTA_MODELOS)
    print("Para treinar pela linha de comando: python -m src.cardioia.ml.train")
    print("Para rodar os testes:               python -m pytest tests -q")

print()
print("=" * 72)
print("Resumo final desta execucao")
print("=" * 72)
print("  base                     :", origem_dados)
print("  sha256 da base           :", dataset.sha256_da_base(df))
print("  features x amostras      :", len(COLUNAS_PREDITORAS), "x", len(df))
print("  modelo salvo             :", execucao["nome_modelo_escolhido"])
for nome, avaliacao in execucao["avaliacoes"].items():
    print(f"  {nome:24s} : acuracia={avaliacao['acuracia']} | recall(1)={round(avaliacao['recall_classe_1'], 4)}")
print(f"  {'Regra deterministica':24s} : acuracia={execucao['acuracia_da_regra']} (teto, por construcao)")
print("  previsao do paciente     :", resultado_paciente["pico_risco_previsto"],
      "(probabilidade", round(resultado_paciente["probabilidade_risco"], 4), ")")
print("  protocolos acionados     :", len(protocolos))
print()
print("AVISO: base sintetica; este modelo nao tem validacao clinica.")